# 02. Ingeniería de Características Geoespaciales y Tabulares

**Proyecto:** Sistema inteligente de estimación de precios de alojamientos turísticos en Málaga  
**Entrada:** `../data/processed/listings_cleaned.parquet`  
**Salida:** `../data/processed/listings_features.parquet`

## Objetivos

1. Convertir los alojamientos en puntos geográficos.
2. Cruzarlos con los distritos municipales mediante un `spatial join`.
3. Calcular superficie, oferta y densidad por distrito.
4. Indexar cada alojamiento en H3 a resoluciones 8 y 9.
5. Calcular competidores y densidad de oferta por hexágono.
6. Calcular distancias geodésicas a playas, centro, cultura y transporte.
7. Ampliar la vectorización de `amenities`.
8. Exportar un dataset enriquecido y auditable.

> Las estadísticas de precio por zona no se calculan en esta fase, ya que utilizan
> directamente la variable objetivo. Estas variables se generarán posteriormente
> dentro del proceso de modelado utilizando únicamente los datos de entrenamiento,
> evitando así fugas de información.

## 0. Instalación de dependencias

Ejecuta esta celda solamente si todavía no tienes las librerías instaladas. Después, reinicia el kernel.

In [ ]:
# Descomenta si es necesario:
%pip install geopandas pyogrio shapely pyproj h3 requests pyarrow

## 1. Librerías, configuración y rutas

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import ast
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import h3

from pyproj import Geod

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

PROCESSED_DIR = Path("../data/processed")
GEOSPATIAL_DIR = Path("../data/geospatial")

INPUT_LISTINGS = PROCESSED_DIR / "listings_cleaned.parquet"

# Se utilizará el GeoJSON local ya descargado.

DISTRICTS_FILE = Path("../data/raw/neighbourhoods.geojson")

OUTPUT_FEATURES = (
    PROCESSED_DIR
    / "listings_features.parquet"
)


OUTPUT_FEATURE_REPORT = (
    PROCESSED_DIR
    / "listings_features_quality_report.csv"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GEOSPATIAL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# PARÁMETROS
# ------------------------------------------------------------

H3_RESOLUTIONS = [8, 9]

# Número máximo de amenities adicionales que se vectorizarán.
TOP_AMENITIES = 60


print(f"Entrada: {INPUT_LISTINGS}")
print("GeoJSON local: neighbourhoods.geojson")
print(f"Salida: {OUTPUT_FEATURES}")

## 2. Comprobación de versiones

In [ ]:
print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("H3:", h3.__version__)

required_h3_functions = [
    "latlng_to_cell",
    "cell_area",
]

missing_h3_functions = [
    function_name
    for function_name in required_h3_functions
    if not hasattr(h3, function_name)
]

if missing_h3_functions:
    raise ImportError(
        "La versión instalada de h3 no tiene la API esperada. "
        f"Faltan: {missing_h3_functions}. "
        "Actualiza con: pip install --upgrade h3"
    )

print("API de H3 compatible.")

## 3. Carga y validación del dataset limpio

In [ ]:
if not INPUT_LISTINGS.exists():
    raise FileNotFoundError(
        f"No existe {INPUT_LISTINGS}. "
        "Ejecuta primero la Fase 1."
    )

df_features = pd.read_parquet(INPUT_LISTINGS)

print(
    f"Dataset cargado: {df_features.shape[0]:,} filas × "
    f"{df_features.shape[1]} columnas"
)

required_columns = [
    "latitude",
    "longitude",
    "price",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_features.columns
]

if missing_columns:
    raise KeyError(
        "Faltan columnas necesarias para la Fase 2: "
        f"{missing_columns}"
    )

df_features["latitude"] = pd.to_numeric(
    df_features["latitude"],
    errors="coerce",
)

df_features["longitude"] = pd.to_numeric(
    df_features["longitude"],
    errors="coerce",
)

df_features["price"] = pd.to_numeric(
    df_features["price"],
    errors="coerce",
)

initial_rows = len(df_features)

df_features = (
    df_features
    .dropna(
        subset=[
            "latitude",
            "longitude",
            "price",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

print(
    "Filas eliminadas por coordenadas o precio nulos: "
    f"{initial_rows - len(df_features):,}"
)

display(df_features.head())

## 4. Auditoría inicial de coordenadas

In [ ]:
coordinate_summary = pd.DataFrame({
    "variable": [
        "latitude",
        "longitude",
        "price",
    ],
    "tipo": [
        str(df_features["latitude"].dtype),
        str(df_features["longitude"].dtype),
        str(df_features["price"].dtype),
    ],
    "nulos": [
        int(df_features["latitude"].isna().sum()),
        int(df_features["longitude"].isna().sum()),
        int(df_features["price"].isna().sum()),
    ],
    "minimo": [
        df_features["latitude"].min(),
        df_features["longitude"].min(),
        df_features["price"].min(),
    ],
    "maximo": [
        df_features["latitude"].max(),
        df_features["longitude"].max(),
        df_features["price"].max(),
    ],
})

display(coordinate_summary)

# Rango amplio de control para Málaga capital y periferia inmediata.
malaga_coordinate_mask = (
    df_features["latitude"].between(36.50, 36.95)
    &
    df_features["longitude"].between(-4.90, -3.95)
)

coordinate_outliers = (
    df_features.loc[
        ~malaga_coordinate_mask,
        [
            "latitude",
            "longitude",
            "price",
        ],
    ]
    .copy()
)

print(
    "Coordenadas potencialmente fuera del entorno de Málaga: "
    f"{len(coordinate_outliers):,}"
)

if not coordinate_outliers.empty:
    display(coordinate_outliers.head(20))

## 5. Conversión a GeoDataFrame

In [ ]:
gdf_listings = gpd.GeoDataFrame(
    df_features.copy(),
    geometry=gpd.points_from_xy(
        df_features["longitude"],
        df_features["latitude"],
    ),
    crs="EPSG:4326",
)

print("GeoDataFrame de alojamientos creado.")
print("CRS:", gdf_listings.crs)
print("Geometrías nulas:", int(gdf_listings.geometry.isna().sum()))

display(
    gdf_listings[
        [
            "latitude",
            "longitude",
            "price",
            "geometry",
        ]
    ].head()
)

## 6. Visualización básica de los puntos

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

gdf_listings.plot(
    ax=ax,
    markersize=4,
    alpha=0.35,
)

ax.set_title("Distribución espacial de los alojamientos")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")

plt.show()

## 7. Carga del GeoJSON local de distritos

Se utiliza el archivo `neighbourhoods.geojson` situado en `data/raw/`,
correspondiente a los datos geográficos descargados de Inside Airbnb. El notebook busca automáticamente en `data/geospatial`, `data` y `data/raw`.


In [ ]:
# ============================================================
# CARGA DEL GEOJSON LOCAL
# ============================================================

DISTRICTS_FILE = next(
    (
        path
        for path in DISTRICTS_FILE_CANDIDATES
        if path.exists()
    ),
    None,
)

if DISTRICTS_FILE is None:
    raise FileNotFoundError(
        "No se ha encontrado neighbourhoods.geojson. "
        "Guárdalo en una de estas rutas:\n"
        + "\n".join(
            f"- {path}"
            for path in DISTRICTS_FILE_CANDIDATES
        )
    )

print(f"GeoJSON local encontrado: {DISTRICTS_FILE}")

gdf_districts_raw = gpd.read_file(
    DISTRICTS_FILE
)

print(
    f"Distritos cargados: "
    f"{gdf_districts_raw.shape[0]:,} filas × "
    f"{gdf_districts_raw.shape[1]} columnas"
)

print("CRS original:", gdf_districts_raw.crs)
print("Columnas disponibles:")
print(gdf_districts_raw.columns.tolist())

display(gdf_districts_raw.head())


## 8. Detección del nombre del distrito y preparación de polígonos

In [ ]:
def normalize_column_name(column_name):
    text = str(column_name).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")


normalized_column_map = {
    normalize_column_name(column): column
    for column in gdf_districts_raw.columns
}

district_name_candidates = [
    "neighbourhood",
    "neighbourhood_group",
    "nombre",
    "distrito",
    "nom_distrito",
    "nombre_distrito",
    "descrip",
    "descripcion",
    "name",
]

DISTRICT_NAME_COLUMN = None

for candidate in district_name_candidates:
    if candidate in normalized_column_map:
        DISTRICT_NAME_COLUMN = (
            normalized_column_map[candidate]
        )
        break

if DISTRICT_NAME_COLUMN is None:
    object_candidates = [
        column
        for column in gdf_districts_raw.columns
        if column != gdf_districts_raw.geometry.name
        and (
            pd.api.types.is_object_dtype(
                gdf_districts_raw[column]
            )
            or pd.api.types.is_string_dtype(
                gdf_districts_raw[column]
            )
        )
    ]

    if not object_candidates:
        raise KeyError(
            "No se ha podido detectar automáticamente "
            "la columna con el nombre del distrito."
        )

    DISTRICT_NAME_COLUMN = object_candidates[0]

print(
    "Columna seleccionada como nombre del distrito: "
    f"{DISTRICT_NAME_COLUMN}"
)

gdf_districts = (
    gdf_districts_raw[
        [
            DISTRICT_NAME_COLUMN,
            gdf_districts_raw.geometry.name,
        ]
    ]
    .rename(
        columns={
            DISTRICT_NAME_COLUMN: "district"
        }
    )
    .copy()
)

gdf_districts["district"] = (
    gdf_districts["district"]
    .astype(str)
    .str.strip()
)

gdf_districts = (
    gdf_districts
    .dropna(subset=["geometry"])
    .drop_duplicates(subset=["district"])
    .reset_index(drop=True)
)

if gdf_districts.crs is None:
    gdf_districts = gdf_districts.set_crs(
        "EPSG:4326"
    )

# El archivo neighbourhoods.geojson puede venir en EPSG:4979
# (WGS84 tridimensional). Para el spatial join se transforma a
# EPSG:4326 bidimensional.
gdf_districts_4326 = (
    gdf_districts
    .to_crs("EPSG:4326")
)

# Forzar geometrías 2D por compatibilidad.
gdf_districts_4326["geometry"] = (
    gdf_districts_4326.geometry
    .apply(
        lambda geometry: (
            geometry
            if not getattr(geometry, "has_z", False)
            else __import__("shapely").force_2d(geometry)
        )
    )
)

display(gdf_districts_4326.head())

## 9. Cálculo de superficie de los distritos

In [ ]:
# EPSG:25830 es UTM 30N y permite calcular superficies en metros.
gdf_districts_metric = (
    gdf_districts_4326
    .to_crs("EPSG:25830")
    .copy()
)

gdf_districts_metric["district_area_km2"] = (
    gdf_districts_metric.geometry.area
    / 1_000_000
)

district_area = (
    gdf_districts_metric[
        [
            "district",
            "district_area_km2",
        ]
    ]
    .copy()
)

print("Superficie por distrito:")
display(
    district_area.sort_values(
        "district_area_km2",
        ascending=False,
    )
)

## 10. Spatial join entre alojamientos y distritos

In [ ]:
gdf_joined = gpd.sjoin(
    gdf_listings,
    gdf_districts_4326[
        [
            "district",
            "geometry",
        ]
    ],
    how="left",
    predicate="within",
)

# Un punto en el límite puede no entrar con within.
# Se intenta un segundo cruce con intersects para los no asignados.
missing_district_mask = (
    gdf_joined["district"].isna()
)

if missing_district_mask.any():
    boundary_points = (
        gdf_listings.loc[
            missing_district_mask
        ]
        .copy()
    )

    boundary_join = gpd.sjoin(
        boundary_points,
        gdf_districts_4326[
            [
                "district",
                "geometry",
            ]
        ],
        how="left",
        predicate="intersects",
    )

    gdf_joined.loc[
        missing_district_mask,
        "district",
    ] = boundary_join["district"].values

gdf_joined = (
    gdf_joined
    .drop(
        columns=["index_right"],
        errors="ignore",
    )
    .reset_index(drop=True)
)

print(
    "Alojamientos sin distrito asignado: "
    f"{gdf_joined['district'].isna().sum():,}"
)

display(
    gdf_joined[
        [
            "latitude",
            "longitude",
            "district",
            "price",
        ]
    ].head()
)

## 11. Oferta y densidad por distrito

In [ ]:
district_listing_counts = (
    gdf_joined
    .groupby(
        "district",
        dropna=False,
    )
    .size()
    .rename("district_listing_count")
    .reset_index()
)

district_features = (
    district_listing_counts
    .merge(
        district_area,
        on="district",
        how="left",
    )
)

district_features[
    "district_listing_density_km2"
] = (
    district_features[
        "district_listing_count"
    ]
    /
    district_features[
        "district_area_km2"
    ]
)

gdf_joined = (
    gdf_joined
    .merge(
        district_features,
        on="district",
        how="left",
    )
)

print("Resumen macroespacial:")
display(
    district_features.sort_values(
        "district_listing_density_km2",
        ascending=False,
    )
)

## 12. Indexación H3

Se crean índices H3 en resoluciones 8 y 9. Las variables seguras para esta fase son:

- número de alojamientos de la celda;
- número de competidores, excluyendo el propio alojamiento;
- superficie de la celda;
- densidad de oferta.

No se calcula aquí el precio medio por H3 porque debe aprenderse exclusivamente con `train`.

In [ ]:
for resolution in H3_RESOLUTIONS:
    h3_column = f"h3_res{resolution}"

    gdf_joined[h3_column] = [
        h3.latlng_to_cell(
            latitude,
            longitude,
            resolution,
        )
        for latitude, longitude in zip(
            gdf_joined["latitude"],
            gdf_joined["longitude"],
        )
    ]

    cell_counts = (
        gdf_joined[h3_column]
        .value_counts()
    )

    count_column = (
        f"h3_res{resolution}_listing_count"
    )

    competitor_column = (
        f"h3_res{resolution}_competitor_count"
    )

    area_column = (
        f"h3_res{resolution}_area_km2"
    )

    density_column = (
        f"h3_res{resolution}_density_km2"
    )

    gdf_joined[count_column] = (
        gdf_joined[h3_column]
        .map(cell_counts)
        .astype("int32")
    )

    gdf_joined[competitor_column] = (
        gdf_joined[count_column] - 1
    ).clip(lower=0).astype("int32")

    unique_cells = (
        gdf_joined[h3_column]
        .drop_duplicates()
    )

    cell_area_map = {
        cell: h3.cell_area(
            cell,
            unit="km^2",
        )
        for cell in unique_cells
    }

    gdf_joined[area_column] = (
        gdf_joined[h3_column]
        .map(cell_area_map)
        .astype("float64")
    )

    gdf_joined[density_column] = (
        gdf_joined[count_column]
        /
        gdf_joined[area_column]
    )

    print(
        f"H3 resolución {resolution}: "
        f"{gdf_joined[h3_column].nunique():,} celdas."
    )

## 13. Resumen de la concentración H3

In [ ]:
h3_summary_rows = []

for resolution in H3_RESOLUTIONS:
    h3_column = f"h3_res{resolution}"
    count_column = (
        f"h3_res{resolution}_listing_count"
    )
    competitor_column = (
        f"h3_res{resolution}_competitor_count"
    )

    h3_summary_rows.append({
        "resolucion": resolution,
        "celdas_unicas": int(
            gdf_joined[h3_column].nunique()
        ),
        "alojamientos_medios_por_celda": round(
            gdf_joined
            .groupby(h3_column)
            .size()
            .mean(),
            3,
        ),
        "maximo_alojamientos_celda": int(
            gdf_joined[count_column].max()
        ),
        "competidores_medios": round(
            gdf_joined[competitor_column].mean(),
            3,
        ),
    })

h3_summary = pd.DataFrame(
    h3_summary_rows
)

display(h3_summary)

## 14. Distancias geodésicas a puntos de interés

Las coordenadas se centralizan en un diccionario para que puedan revisarse o actualizarse fácilmente. Las distancias se calculan con el elipsoide WGS84 y se expresan en kilómetros.

In [ ]:

POIS = {
    # Playas
    "malagueta": (
        36.7196,
        -4.4087,
        "beach",
    ),
    "pedregalejo": (
        36.7209,
        -4.3699,
        "beach",
    ),
    "el_palo": (
        36.7169,
        -4.3565,
        "beach",
    ),

    # Centro histórico
    "calle_larios": (
        36.7196,
        -4.4211,
        "historic_center",
    ),
    "plaza_constitucion": (
        36.7202,
        -4.4218,
        "historic_center",
    ),

    # Cultura
    "museo_picasso": (
        36.7217,
        -4.4185,
        "culture",
    ),
    "alcazaba": (
        36.7213,
        -4.4165,
        "culture",
    ),
    "centro_pompidou": (
        36.7182,
        -4.4130,
        "culture",
    ),

    # Transporte
    "maria_zambrano": (
        36.7111,
        -4.4314,
        "transport",
    ),
    "aeropuerto_agp": (
        36.6749,
        -4.4991,
        "transport",
    ),
}

poi_table = pd.DataFrame(
    [
        {
            "poi": poi_name,
            "latitude": values[0],
            "longitude": values[1],
            "category": values[2],
        }
        for poi_name, values in POIS.items()
    ]
)

display(poi_table)

In [ ]:
geod = Geod(ellps="WGS84")

listing_longitudes = (
    gdf_joined["longitude"]
    .to_numpy(dtype=float)
)

listing_latitudes = (
    gdf_joined["latitude"]
    .to_numpy(dtype=float)
)

for poi_name, (
    poi_latitude,
    poi_longitude,
    poi_category,
) in POIS.items():

    _, _, distances_meters = geod.inv(
        listing_longitudes,
        listing_latitudes,
        np.full(
            len(gdf_joined),
            poi_longitude,
            dtype=float,
        ),
        np.full(
            len(gdf_joined),
            poi_latitude,
            dtype=float,
        ),
    )

    gdf_joined[
        f"distance_{poi_name}_km"
    ] = (
        distances_meters
        / 1000
    )

print("Distancias individuales calculadas.")

## 15. Distancias mínimas por categoría de POI

In [ ]:
poi_categories = sorted({
    values[2]
    for values in POIS.values()
})

for category in poi_categories:
    category_pois = [
        poi_name
        for poi_name, values in POIS.items()
        if values[2] == category
    ]

    category_distance_columns = [
        f"distance_{poi_name}_km"
        for poi_name in category_pois
    ]

    gdf_joined[
        f"distance_nearest_{category}_km"
    ] = (
        gdf_joined[
            category_distance_columns
        ]
        .min(axis=1)
    )

    gdf_joined[
        f"nearest_{category}_poi"
    ] = (
        gdf_joined[
            category_distance_columns
        ]
        .idxmin(axis=1)
        .str.replace(
            "distance_",
            "",
            regex=False,
        )
        .str.replace(
            "_km",
            "",
            regex=False,
        )
    )

print("Distancias mínimas y POIs más cercanos calculados.")

nearest_columns = [
    column
    for column in gdf_joined.columns
    if column.startswith("distance_nearest_")
]

display(
    gdf_joined[
        nearest_columns
    ].describe().T
)

## 16. Preparación y ampliación de amenities

La Fase 1 ya contiene indicadores manuales importantes. Aquí se añaden automáticamente las amenities más frecuentes que todavía no estén representadas, hasta alcanzar `TOP_AMENITIES`.

Los nombres se normalizan para crear columnas válidas y reproducibles.

In [ ]:
def parse_amenities(value):
    """
    Convierte amenities en una lista limpia de strings,
    independientemente de que llegue como lista, array,
    tupla, texto o valor nulo.
    """

    # Listas, tuplas, sets o arrays de NumPy
    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    # Valores nulos escalares
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if value is pd.NA:
        return []

    # Texto
    if isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        try:
            parsed = ast.literal_eval(value)

            if isinstance(
                parsed,
                (
                    list,
                    tuple,
                    set,
                    np.ndarray,
                ),
            ):
                return [
                    str(item).strip()
                    for item in parsed
                    if str(item).strip()
                ]

        except (
            ValueError,
            SyntaxError,
        ):
            pass

        return [
            item.strip()
            for item in value.split(",")
            if item.strip()
        ]

    # Cualquier otro tipo
    return []

def slugify_amenity(text):
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text,
    ).strip("_")

    return text[:60]


if "amenities_list" in gdf_joined.columns:
    amenities_series = (
        gdf_joined["amenities_list"]
        .apply(parse_amenities)
    )

elif "amenities" in gdf_joined.columns:
    amenities_series = (
        gdf_joined["amenities"]
        .apply(parse_amenities)
    )

else:
    amenities_series = pd.Series(
        [[] for _ in range(len(gdf_joined))],
        index=gdf_joined.index,
    )

gdf_joined["amenities_count_phase2"] = (
    amenities_series
    .str.len()
    .astype("int16")
)

all_amenities = (
    amenities_series
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

amenity_frequency = (
    all_amenities[
        all_amenities.ne("")
    ]
    .value_counts()
)

top_amenities = (
    amenity_frequency
    .head(TOP_AMENITIES)
    .index
    .tolist()
)

existing_columns = set(
    gdf_joined.columns
)

generated_amenity_columns = []
amenity_column_map = {}

for amenity in top_amenities:
    base_column = (
        "amenity_auto_"
        + slugify_amenity(amenity)
    )

    column_name = base_column
    suffix = 2

    while (
        column_name in existing_columns
        or column_name in generated_amenity_columns
    ):
        column_name = (
            f"{base_column}_{suffix}"
        )
        suffix += 1

    amenity_column_map[amenity] = column_name
    generated_amenity_columns.append(column_name)

amenity_sets = amenities_series.apply(
    lambda values: set(
        str(value).strip()
        for value in values
    )
)

for amenity, column_name in amenity_column_map.items():
    gdf_joined[column_name] = (
        amenity_sets
        .apply(
            lambda values: int(
                amenity in values
            )
        )
        .astype("int8")
    )

print(
    "Amenities automáticas generadas: "
    f"{len(generated_amenity_columns)}"
)

display(
    pd.DataFrame({
        "amenity_original": list(
            amenity_column_map.keys()
        ),
        "columna_generada": list(
            amenity_column_map.values()
        ),
        "frecuencia": [
            int(amenity_frequency[amenity])
            for amenity in amenity_column_map
        ],
    }).head(25)
)

## 17. Variables adicionales derivadas de distancias y localización

Se añaden indicadores sencillos y explicables útiles para los modelos.

In [ ]:
distance_thresholds = {
    "within_500m_beach": (
        "distance_nearest_beach_km",
        0.5,
    ),
    "within_1km_beach": (
        "distance_nearest_beach_km",
        1.0,
    ),
    "within_1km_historic_center": (
        "distance_nearest_historic_center_km",
        1.0,
    ),
    "within_2km_historic_center": (
        "distance_nearest_historic_center_km",
        2.0,
    ),
    "within_1km_culture": (
        "distance_nearest_culture_km",
        1.0,
    ),
    "within_1km_transport": (
        "distance_nearest_transport_km",
        1.0,
    ),
    "within_5km_airport": (
        "distance_aeropuerto_agp_km",
        5.0,
    ),
}

for new_column, (
    distance_column,
    threshold,
) in distance_thresholds.items():

    if distance_column in gdf_joined.columns:
        gdf_joined[new_column] = (
            gdf_joined[distance_column]
            <= threshold
        ).astype("int8")

# Transformaciones suaves para distancias.
distance_columns = [
    column
    for column in gdf_joined.columns
    if column.startswith("distance_")
    and column.endswith("_km")
]

for column in distance_columns:
    gdf_joined[
        f"log1p_{column}"
    ] = np.log1p(
        gdf_joined[column]
    )

print(
    "Indicadores de proximidad y "
    "transformaciones logarítmicas creados."
)

## 18. Preparación del dataset final

Se elimina la geometría y los textos brutos de amenities. Se conservan:

- categorías legibles;
- identificadores H3;
- nombres de POI más cercanos;
- variables numéricas enriquecidas.

Las columnas categóricas se codificarán dentro del pipeline de modelado.

In [ ]:
df_final = pd.DataFrame(
    gdf_joined.drop(
        columns=[
            "geometry",
            "amenities",
            "amenities_list",
        ],
        errors="ignore",
    )
).copy()

# Evitar columnas creadas accidentalmente por cruces espaciales.
df_final = df_final.drop(
    columns=[
        column
        for column in df_final.columns
        if column.startswith("index_")
    ],
    errors="ignore",
)

# Comprobar nombres duplicados.
duplicated_column_names = (
    df_final.columns[
        df_final.columns.duplicated()
    ]
    .tolist()
)

if duplicated_column_names:
    raise ValueError(
        "Existen nombres de columnas duplicados: "
        f"{duplicated_column_names}"
    )

print(
    "Dataset enriquecido preparado: "
    f"{df_final.shape[0]:,} filas × "
    f"{df_final.shape[1]} columnas"
)

if df_final.shape[1] < 100:
    print(
        "El dataset todavía tiene menos de 100 columnas. "
        "Puedes aumentar TOP_AMENITIES si necesitas una "
        "vectorización más amplia."
    )
else:
    print(
        "El dataset supera las 100 características."
    )

## 19. Control de calidad final

In [ ]:
def classify_feature(series):
    unique_values = set(
        series.dropna().unique().tolist()
    )

    if series.name == "price":
        return "Objetivo"

    if len(unique_values) == 1:
        return "Constante"

    if unique_values.issubset(
        {0, 1, False, True}
    ):
        return "Binaria"

    if pd.api.types.is_integer_dtype(series):
        return "Numérica discreta"

    if pd.api.types.is_float_dtype(series):
        return "Numérica continua"

    return "Categórica"


quality_report = pd.DataFrame({
    "variable": df_final.columns,
    "tipo_dato": (
        df_final.dtypes
        .astype(str)
        .values
    ),
    "tipo_variable": [
        classify_feature(
            df_final[column]
        )
        for column in df_final.columns
    ],
    "nulos": (
        df_final
        .isna()
        .sum()
        .values
    ),
    "porcentaje_nulos": (
        df_final
        .isna()
        .mean()
        .mul(100)
        .round(4)
        .values
    ),
    "valores_unicos": [
        int(
            df_final[column]
            .nunique(dropna=False)
        )
        for column in df_final.columns
    ],
})

summary_final = pd.DataFrame({
    "metrica": [
        "Filas",
        "Columnas",
        "Nulos totales",
        "Filas duplicadas",
        "Variables binarias",
        "Variables continuas",
        "Variables discretas",
        "Variables categóricas",
        "Variables constantes",
        "Memoria MB",
    ],
    "valor": [
        len(df_final),
        df_final.shape[1],
        int(df_final.isna().sum().sum()),
        int(df_final.duplicated().sum()),
        int(
            (
                quality_report["tipo_variable"]
                == "Binaria"
            ).sum()
        ),
        int(
            (
                quality_report["tipo_variable"]
                == "Numérica continua"
            ).sum()
        ),
        int(
            (
                quality_report["tipo_variable"]
                == "Numérica discreta"
            ).sum()
        ),
        int(
            (
                quality_report["tipo_variable"]
                == "Categórica"
            ).sum()
        ),
        int(
            (
                quality_report["tipo_variable"]
                == "Constante"
            ).sum()
        ),
        round(
            df_final
            .memory_usage(deep=True)
            .sum()
            / 1024**2,
            2,
        ),
    ],
})

display(summary_final)

print("\nVariables con nulos:")
variables_with_nulls = (
    quality_report[
        quality_report["nulos"] > 0
    ]
    .sort_values(
        "porcentaje_nulos",
        ascending=False,
    )
)

if variables_with_nulls.empty:
    print("No existen variables con nulos.")
else:
    display(variables_with_nulls)

print("\nDistribución de tipos:")
display(
    quality_report[
        "tipo_variable"
    ]
    .value_counts()
    .rename_axis("tipo_variable")
    .reset_index(name="numero_variables")
)

## 20. Comprobaciones específicas de fuga de información

Las columnas que contienen agregaciones de `price` no deben existir todavía.

In [ ]:
potential_target_leakage_columns = [
    column
    for column in df_final.columns
    if (
        "mean_price" in column.lower()
        or "median_price" in column.lower()
        or "avg_price" in column.lower()
        or "price_encoding" in column.lower()
    )
]

print(
    "Posibles columnas de fuga por agregación del target: "
    f"{potential_target_leakage_columns}"
)

if potential_target_leakage_columns:
    raise ValueError(
        "Se han detectado posibles agregaciones de price "
        "antes de dividir train y test."
    )

print(
    "No se han incorporado medias de price "
    "por distrito o H3."
)

## 21. Exportación

In [ ]:
# ============================================================
# EXPORTACIÓN DEL DATASET FINAL DE CARACTERÍSTICAS
# ============================================================

df_final.to_parquet(
    OUTPUT_FEATURES,
    index=False,
)

print(
    f"Dataset exportado: {df_final.shape[0]:,} filas × "
    f"{df_final.shape[1]} columnas"
)
print(f"Ruta: {OUTPUT_FEATURES}")

## 22. Verificación de la exportación

In [ ]:
df_features_check = pd.read_parquet(
    OUTPUT_FEATURES
)

print(
    "Archivo recargado correctamente: "
    f"{df_features_check.shape[0]:,} filas × "
    f"{df_features_check.shape[1]} columnas"
)

assert (
    df_features_check.shape
    == df_final.shape
)

assert (
    df_features_check.columns.tolist()
    == df_final.columns.tolist()
)

print("Exportación verificada.")

preview_columns = [
    column
    for column in [
        "price",
        "latitude",
        "longitude",
        "district",
        "district_listing_density_km2",
        "h3_res8_competitor_count",
        "h3_res9_competitor_count",
        "distance_nearest_beach_km",
        "distance_nearest_historic_center_km",
        "distance_aeropuerto_agp_km",
        "amenities_count_phase2",
    ]
    if column in df_features_check.columns
]

display(
    df_features_check[
        preview_columns
    ].head()
)

## 23. Visualización de las características geoespaciales

Como apoyo al análisis, se representan tres elementos utilizados durante la
ingeniería de características: la distribución territorial de los alojamientos,
la división del espacio mediante celdas H3 y un ejemplo del entorno competitivo
definido alrededor de un alojamiento.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# --- Cargar alojamientos ---
df = pd.read_parquet("../data/processed/listings_cleaned.parquet")

# Convertir a GeoDataFrame
gdf_listings = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# --- Cargar barrios ---
gdf_barrios = gpd.read_file("../data/raw/neighbourhoods.geojson")

# Asegurar mismo CRS
gdf_barrios = gdf_barrios.to_crs(gdf_listings.crs)

# --- Dibujar ---
fig, ax = plt.subplots(figsize=(8, 8))

gdf_barrios.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.8
)

gdf_listings.sample(
    min(1000, len(gdf_listings)),
    random_state=42
).plot(
    ax=ax,
    markersize=4,
    alpha=0.45
)


ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import h3

# ============================================================
# 1. CARGAR DATOS
# ============================================================

df = pd.read_parquet("../data/processed/listings_features.parquet")

# GeoDataFrame de alojamientos
gdf_listings = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# ============================================================
# 2. CONVERTIR CELDAS H3 A POLÍGONOS
# ============================================================

def h3_to_polygon(h):
    boundary = h3.cell_to_boundary(h)
    # H3 devuelve (lat, lon), Shapely necesita (lon, lat)
    return Polygon([(lon, lat) for lat, lon in boundary])

h3_cells = df["h3_res8"].dropna().unique()

gdf_h3 = gpd.GeoDataFrame(
    {"h3_res8": h3_cells},
    geometry=[h3_to_polygon(h) for h in h3_cells],
    crs="EPSG:4326"
)

# ============================================================
# 3. LÍMITES DEL ZOOM
#    Málaga urbana + litoral
# ============================================================

min_lon, max_lon = -4.50, -4.34
min_lat, max_lat = 36.68, 36.76

# Alojamientos visibles en el zoom
gdf_zoom = gdf_listings[
    gdf_listings["longitude"].between(min_lon, max_lon) &
    gdf_listings["latitude"].between(min_lat, max_lat)
]

# Celdas H3 que intersectan la zona visible
gdf_h3_zoom = gdf_h3.cx[min_lon:max_lon, min_lat:max_lat]

# ============================================================
# 4. REPRESENTACIÓN
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

# Celdas H3
gdf_h3_zoom.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.7,
    alpha=0.8
)

# Alojamientos
gdf_zoom.plot(
    ax=ax,
    markersize=7,
    alpha=0.45
)

# Límites exactos del zoom
ax.set_xlim(min_lon, max_lon)
ax.set_ylim(min_lat, max_lat)


ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# Cargar datos
df = pd.read_parquet("../data/processed/listings_features.parquet")

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# Pasar a CRS métrico
gdf_m = gdf.to_crs("EPSG:25830")

# Elegir un alojamiento relativamente céntrico
ejemplo = gdf_m.iloc[len(gdf_m) // 2]

# Radio de 1000 metros
radio = ejemplo.geometry.buffer(1000)

# Alojamientos dentro del radio
competidores = gdf_m[gdf_m.geometry.within(radio)]

fig, ax = plt.subplots(figsize=(8, 8))

# Radio
gpd.GeoSeries([radio], crs=gdf_m.crs).plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1.5,
    linestyle="--"
)

# Competidores
competidores.plot(
    ax=ax,
    markersize=15,
    alpha=0.5
)

# Alojamiento seleccionado
gpd.GeoSeries(
    [ejemplo.geometry],
    crs=gdf_m.crs
).plot(
    ax=ax,
    markersize=90,
    marker="*"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

print("Alojamientos en 1 km:", len(competidores) - 1)

## 24. Cierre de la fase

Tras incorporar las variables tabulares y geoespaciales se obtiene el dataset
`listings_features.parquet`, que se utilizará como entrada del motor predictivo.

El conjunto final incluye las características originales seleccionadas, variables
derivadas del alojamiento, información territorial basada en H3, medidas de
competencia local y distancias a puntos de interés.

En la siguiente fase se realizará la validación espacial, el entrenamiento de los
modelos y la selección del modelo predictivo final.